In [1]:
import tensorflow as tf
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

In [2]:
training_set = tf.keras.preprocessing.image_dataset_from_directory(
    "Crop Diseases",
    labels="inferred",
    label_mode="categorical",
    class_names= None,
    color_mode="rgb",
    batch_size=64,
    image_size=(180, 180),
    shuffle=True,
    seed=42,
    validation_split=0.2,
    subset="training",
    interpolation="bilinear",
    follow_links=False,
    crop_to_aspect_ratio=False,
)

Found 13324 files belonging to 17 classes.
Using 10660 files for training.


2026-09-06 18:46:49.991298: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M5
2026-09-06 18:46:49.991337: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 16.00 GB
2026-09-06 18:46:49.991354: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 5.92 GB
2026-09-06 18:46:49.991367: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2026-09-06 18:46:49.991378: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


In [3]:
validation_set = tf.keras.preprocessing.image_dataset_from_directory(
    "Crop Diseases",
    labels="inferred",
    label_mode="categorical",
    class_names= None,
    color_mode="rgb",
    batch_size=64,
    image_size=(180, 180),
    shuffle=True,
    seed=42,
    validation_split=0.2,
    subset="validation",
    interpolation="bilinear",
    follow_links=False,
    crop_to_aspect_ratio=False,
)

Found 13324 files belonging to 17 classes.
Using 2664 files for validation.


In [4]:
AUTOTUNE = tf.data.AUTOTUNE
training_set = training_set.cache().prefetch(buffer_size=AUTOTUNE)
validation_set = validation_set.cache().prefetch(buffer_size=AUTOTUNE)

In [5]:
import ssl
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping

In [6]:

ssl._create_default_https_context = ssl._create_unverified_context

# 1. Base model
base_model = MobileNetV2(
    input_shape=(180, 180, 3),
    include_top=False,
    weights='imagenet'
)

# 2. Pipeline with explicit MobileNet scaling [-1, 1]
inputs = tf.keras.Input(shape=(180, 180, 3))

x = layers.RandomFlip("horizontal_and_vertical")(inputs)
x = layers.RandomRotation(0.15)(x)
x = layers.RandomZoom(0.1)(x)

x = preprocess_input(inputs) 
x = base_model(x, training=False)  # Locks BatchNormalization in inference mode
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.5)(x)

outputs = layers.Dense(17, activation='softmax', kernel_regularizer=tf.keras.regularizers.l2(0.01))(x)

model = tf.keras.Model(inputs, outputs)

# 3. Unfreeze
base_model.trainable = True

# 4. Compile with fine-tuning learning rate
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=3e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)


/var/folders/3g/gz14xqys2873zm_jkz97fysc0000gn/T/ipykernel_74101/2961241854.py:4: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  base_model = MobileNetV2(


In [7]:
callbacks = [
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=0.000001, verbose=1),
    EarlyStopping(monitor='val_loss', patience=6, restore_best_weights=True, verbose=1)
]

In [ ]:

# 5. Fit model
history = model.fit(
    x=training_set,
    validation_data=validation_set,
    epochs=30,
    callbacks=callbacks
)

Epoch 1/30


2026-09-06 18:46:54.050953: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.


167/167 ━━━━━━━━━━━━━━━━━━━━ 161s 867ms/step - accuracy: 0.6083 - loss: 1.6310 - val_accuracy: 0.6622 - val_loss: 1.3781 - learning_rate: 3.0000e-05
Epoch 2/30
 24/167 ━━━━━━━━━━━━━━━━━━━━ 1:42 717ms/step - accuracy: 0.8273 - loss: 0.8635